### IDX Exchange Team **ds55**
### Beini Lan
# 06_evaluation - Week 8 Evaluation Expansion

This notebook evaluates all five model families explored in prior weeks on one
common, current time split:

- **Training:** June 2025 through May 2026 (12 complete months)
- **Held-out test:** June 2026
- **Models:** Linear Regression, Decision Tree, Random Forest, XGBoost, and LightGBM
- **Metrics:** test R², MAPE, MdAPE, MSE, and RMSE (with MAE retained as an
  additional diagnostic)
- **Price-band diagnostic:** five equal-frequency June 2026 price bands,
  evaluated separately for every model

W7 feature pipeline is reused. Previously selected model settings are frozen; June 2026 is used only once for test evaluation.

## Setup and reproducibility controls

All models use the same fixed seed, leakage exclusions, engineered predictors,
preprocessing fit, training rows, and untouched test rows. Model settings come
from the prior-week work so Week 8 measures out-of-time performance rather than
using the holdout for tuning.

In [1]:
from pathlib import Path
import json
import os
import random
import time
import warnings

import geopandas as gpd
import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:,.4f}".format)

TEAM = "Team ds55, Beini Lan"
RANDOM_STATE = 55
N_JOBS = -1
TRAIN_START_MONTH = "2025-06"
TRAIN_END_MONTH = "2026-05"
TEST_MONTH = "2026-06"
FORBIDDEN_FEATURE_TERMS = [
    "ListPrice",
    "OriginalListPrice",
    "DaysOnMarket",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate",
    "PricePerLivingArea",
]

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    env_root = os.environ.get("IDX_PROJECT_ROOT")
    if env_root:
        candidates.insert(0, Path(env_root).expanduser())

    for candidate in candidates:
        if (
            (candidate / "raw data" / "CRMLSSold202606.csv").exists()
            and (candidate / "W6 Feature Engineering").exists()
        ):
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate the IDX Exchange project root. "
        "Run from the project or set IDX_PROJECT_ROOT."
    )


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "raw data"
OUTPUT_DIR = PROJECT_ROOT / "W8 Evaluation Expansion"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCHOOL_DISTRICT_FILE = (
    PROJECT_ROOT
    / "W6 Feature Engineering"
    / "resources"
    / "California_School_District_Areas_2025-26.geojson"
)

environment_summary = pd.DataFrame(
    [
        {"item": "project root", "value": str(PROJECT_ROOT)},
        {"item": "pandas", "value": pd.__version__},
        {"item": "scikit-learn", "value": __import__("sklearn").__version__},
        {"item": "XGBoost", "value": xgb.__version__},
        {"item": "LightGBM", "value": lgb.__version__},
        {"item": "random seed", "value": RANDOM_STATE},
    ]
)
display(environment_summary)

,item,value
0,project root,/Users/HP/Documents/IDX Exchange
1,pandas,2.3.3
2,scikit-learn,1.7.1
3,XGBoost,3.3.0
4,LightGBM,4.7.0
5,random seed,55


## Import the shifted monthly raw data

The required inputs are the 13 monthly files from `CRMLSSold202506.csv` through
`CRMLSSold202606.csv`. Source-file month and `CloseDate` month are cross-checked
to prevent a mislabeled or mixed-period file from silently entering the
experiment.

In [2]:
RAW_COLUMNS = [
    "StateOrProvince",
    "PropertyType",
    "PropertySubType",
    "CloseDate",
    "ClosePrice",
    "Flooring",
    "Levels",
    "ViewYN",
    "PoolPrivateYN",
    "Latitude",
    "Longitude",
    "LivingArea",
    "AttachedGarageYN",
    "ParkingTotal",
    "YearBuilt",
    "BathroomsTotalInteger",
    "BedroomsTotal",
    "FireplaceYN",
    "MainLevelBedrooms",
    "NewConstructionYN",
    "GarageSpaces",
    "HighSchoolDistrict",
    "PostalCode",
    "AssociationFee",
    "LotSizeSquareFeet",
]
NUMERIC_RAW_COLUMNS = [
    "ClosePrice",
    "Latitude",
    "Longitude",
    "LivingArea",
    "ParkingTotal",
    "YearBuilt",
    "BathroomsTotalInteger",
    "BedroomsTotal",
    "MainLevelBedrooms",
    "GarageSpaces",
    "AssociationFee",
    "LotSizeSquareFeet",
]

expected_periods = pd.period_range(
    TRAIN_START_MONTH,
    TEST_MONTH,
    freq="M",
)
monthly_frames = []
source_rows = []

for period in expected_periods:
    source_month = period.strftime("%Y%m")
    source_path = RAW_DIR / f"CRMLSSold{source_month}.csv"
    if not source_path.exists():
        raise FileNotFoundError(f"Missing required raw file: {source_path}")

    header = pd.read_csv(source_path, nrows=0).columns.str.strip()
    missing_columns = sorted(set(RAW_COLUMNS).difference(header))
    if missing_columns:
        raise ValueError(f"{source_path.name} is missing columns: {missing_columns}")

    frame = pd.read_csv(
        source_path,
        usecols=RAW_COLUMNS,
        dtype={"PostalCode": "string"},
        low_memory=False,
    )
    frame.columns = frame.columns.str.strip()
    frame["SourceFile"] = source_path.name
    frame["SourceMonth"] = str(period)
    monthly_frames.append(frame)
    source_rows.append(
        {
            "source_file": source_path.name,
            "source_month": str(period),
            "raw_rows": len(frame),
        }
    )

raw = pd.concat(monthly_frames, ignore_index=True)
raw["CloseDate"] = pd.to_datetime(raw["CloseDate"], errors="coerce")
for column in NUMERIC_RAW_COLUMNS:
    raw[column] = pd.to_numeric(raw[column], errors="coerce")

date_month = raw["CloseDate"].dt.to_period("M").astype("string")
month_mismatches = date_month.notna() & date_month.ne(raw["SourceMonth"])
if month_mismatches.any():
    raise ValueError(
        f"Found {month_mismatches.sum():,} rows whose CloseDate month "
        "does not match the source-file month."
    )

source_summary = pd.DataFrame(source_rows)
source_summary.loc[:, "ca_sfr_positive_price_rows"] = [
    int(
        (
            raw.loc[raw["SourceFile"].eq(row["source_file"]), "StateOrProvince"].eq("CA")
            & raw.loc[raw["SourceFile"].eq(row["source_file"]), "PropertyType"].eq("Residential")
            & raw.loc[raw["SourceFile"].eq(row["source_file"]), "PropertySubType"].eq(
                "SingleFamilyResidence"
            )
            & raw.loc[raw["SourceFile"].eq(row["source_file"]), "ClosePrice"].gt(0)
        ).sum()
    )
    for row in source_rows
]
display(source_summary)
print(f"Imported {len(raw):,} raw rows from {len(source_summary)} monthly CSV files.")

,source_file,source_month,raw_rows,ca_sfr_positive_price_rows
0,CRMLSSold202506.csv,2025-06,22883,11700
1,CRMLSSold202507.csv,2025-07,23646,12113
2,CRMLSSold202508.csv,2025-08,22972,11452
3,CRMLSSold202509.csv,2025-09,22443,11456
4,CRMLSSold202510.csv,2025-10,23233,12028
5,CRMLSSold202511.csv,2025-11,19088,9739
6,CRMLSSold202512.csv,2025-12,20538,10455
7,CRMLSSold202601.csv,2026-01,16487,7490
8,CRMLSSold202602.csv,2026-02,18124,8550
9,CRMLSSold202603.csv,2026-03,22583,11177


Imported 283,176 raw rows from 13 monthly CSV files.


## Apply the established Week 3 preprocessing

The W7 implementation of the Week 3 cleaning rules is reused: eligible
California single-family sales are retained, invalid physical values become
missing, categorical fields are normalized, and target outlier rules and
continuous caps are learned from training data only. No June 2026 test row is
removed.

In [3]:
row_filter = (
    raw["StateOrProvince"].eq("CA")
    & raw["PropertyType"].eq("Residential")
    & raw["PropertySubType"].eq("SingleFamilyResidence")
    & raw["ClosePrice"].notna()
    & raw["ClosePrice"].gt(0)
)

df = raw.loc[row_filter].copy()
df["SaleMonth"] = df["CloseDate"].dt.to_period("M").astype("string")


def set_invalid_to_missing(frame, column, condition):
    before = int(frame[column].isna().sum())
    frame.loc[condition, column] = np.nan
    return int(frame[column].isna().sum()) - before


invalid_rule_counts = {}
invalid_rule_counts["Latitude"] = set_invalid_to_missing(
    df, "Latitude", ~df["Latitude"].between(32, 42)
)
invalid_rule_counts["Longitude"] = set_invalid_to_missing(
    df, "Longitude", ~df["Longitude"].between(-125, -114)
)
invalid_rule_counts["LotSizeSquareFeet"] = set_invalid_to_missing(
    df, "LotSizeSquareFeet", df["LotSizeSquareFeet"].le(100)
)
invalid_rule_counts["LivingArea"] = set_invalid_to_missing(
    df, "LivingArea", df["LivingArea"].le(0)
)
invalid_rule_counts["ParkingTotal"] = set_invalid_to_missing(
    df, "ParkingTotal", df["ParkingTotal"].lt(0)
)
invalid_rule_counts["GarageSpaces"] = set_invalid_to_missing(
    df, "GarageSpaces", df["GarageSpaces"].lt(0)
)
invalid_rule_counts["BathroomsTotalInteger"] = set_invalid_to_missing(
    df, "BathroomsTotalInteger", df["BathroomsTotalInteger"].le(0)
)
invalid_rule_counts["BedroomsTotal"] = set_invalid_to_missing(
    df, "BedroomsTotal", df["BedroomsTotal"].le(0)
)
invalid_rule_counts["MainLevelBedrooms_negative"] = set_invalid_to_missing(
    df, "MainLevelBedrooms", df["MainLevelBedrooms"].lt(0)
)
invalid_rule_counts["AssociationFee"] = set_invalid_to_missing(
    df, "AssociationFee", df["AssociationFee"].lt(0)
)
invalid_rule_counts["MainLevelBedrooms_consistency"] = set_invalid_to_missing(
    df,
    "MainLevelBedrooms",
    df["MainLevelBedrooms"].gt(df["BedroomsTotal"]),
)

count_upper_rules = {
    "ParkingTotal": 50,
    "GarageSpaces": 20,
    "BedroomsTotal": 20,
    "BathroomsTotalInteger": 20,
    "MainLevelBedrooms": 20,
}
for column, upper_bound in count_upper_rules.items():
    invalid_rule_counts[f"{column}_upper"] = set_invalid_to_missing(
        df, column, df[column].gt(upper_bound)
    )

categorical_inputs = [
    "Flooring",
    "Levels",
    "ViewYN",
    "PoolPrivateYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "NewConstructionYN",
    "HighSchoolDistrict",
    "PostalCode",
]
for column in categorical_inputs:
    df[column] = df[column].astype("string").str.strip()
    df[column] = df[column].replace(r"^\s*$", pd.NA, regex=True)

df["PostalCode"] = df["PostalCode"].str.extract(r"(\d{5})", expand=False)
ambiguous_school_pattern = (
    r"(?i)^(other|see remarks|call listing office|"
    r"more than 1 district.*|.*inquire.*)$"
)
df["HighSchoolDistrict"] = df["HighSchoolDistrict"].replace(
    ambiguous_school_pattern,
    pd.NA,
    regex=True,
)

flooring_clean = (
    df["Flooring"]
    .str.lower()
    .str.replace(r"\s*,\s*", ",", regex=True)
)
df["Flooring_missing"] = df["Flooring"].isna().astype("int8")
flooring_materials = {
    "has_carpet": "carpet",
    "has_tile": "tile",
    "has_wood": "wood",
    "has_vinyl": "vinyl",
    "has_laminate": "laminate",
    "has_stone": "stone",
    "has_concrete": "concrete",
    "has_bamboo": "bamboo",
    "has_brick": "brick",
}
for new_column, token in flooring_materials.items():
    df[new_column] = flooring_clean.str.contains(
        rf"(?:^|,){token}(?:,|$)",
        regex=True,
        na=False,
    ).astype("int8")
df["Flooring_see_remarks"] = flooring_clean.str.contains(
    r"(?:^|,)seeremarks(?:,|$)",
    regex=True,
    na=False,
).astype("int8")

levels_clean = (
    df["Levels"]
    .str.lower()
    .str.replace(r"\s*,\s*", ",", regex=True)
)
df["Levels_missing"] = df["Levels"].isna().astype("int8")
level_patterns = {
    "level_one": "one",
    "level_two": "two",
    "level_three_or_more": "threeormore",
    "level_multisplit": "multisplit",
}
for new_column, token in level_patterns.items():
    df[new_column] = levels_clean.str.contains(
        rf"(?:^|,){token}(?:,|$)",
        regex=True,
        na=False,
    ).astype("int8")

BASE_FEATURE_COLUMNS = [
    "ViewYN",
    "PoolPrivateYN",
    "Latitude",
    "Longitude",
    "LivingArea",
    "AttachedGarageYN",
    "ParkingTotal",
    "YearBuilt",
    "BathroomsTotalInteger",
    "BedroomsTotal",
    "FireplaceYN",
    "MainLevelBedrooms",
    "NewConstructionYN",
    "GarageSpaces",
    "HighSchoolDistrict",
    "PostalCode",
    "AssociationFee",
    "LotSizeSquareFeet",
    "Flooring_missing",
    *flooring_materials.keys(),
    "Flooring_see_remarks",
    "Levels_missing",
    *level_patterns.keys(),
]

df = df[
    [
        "SourceFile",
        "SourceMonth",
        "CloseDate",
        "SaleMonth",
        "ClosePrice",
        *BASE_FEATURE_COLUMNS,
    ]
].copy()
df["split"] = np.where(df["SaleMonth"].eq(TEST_MONTH), "test", "train")

training_mask_before_outliers = df["split"].eq("train")
missing_indicator_sources = [
    column
    for column in BASE_FEATURE_COLUMNS
    if df.loc[training_mask_before_outliers, column].isna().any()
]
for column in missing_indicator_sources:
    df[f"{column}_missing_ind"] = df[column].isna().astype("int8")

train_before_outliers = df.loc[training_mask_before_outliers].copy()
price_per_living_area = (
    train_before_outliers["ClosePrice"] / train_before_outliers["LivingArea"]
).replace([np.inf, -np.inf], np.nan)
low_price_threshold = train_before_outliers["ClosePrice"].quantile(0.001)
top_price_threshold = train_before_outliers["ClosePrice"].quantile(0.999)
top_price_per_area = price_per_living_area.loc[
    train_before_outliers["ClosePrice"].gt(top_price_threshold)
]
q1 = top_price_per_area.quantile(0.25)
q3 = top_price_per_area.quantile(0.75)
iqr = q3 - q1
price_per_area_cutoff = (
    q3 + 1.5 * iqr
    if pd.notna(iqr) and iqr > 0
    else np.inf
)
remove_train_outlier = (
    train_before_outliers["ClosePrice"].lt(low_price_threshold)
    | price_per_living_area.gt(price_per_area_cutoff)
)
removed_train_indices = train_before_outliers.index[remove_train_outlier]
df = df.drop(index=removed_train_indices).reset_index(drop=True)

train_mask = df["split"].eq("train")
cap_values = {}
for column in ["LivingArea", "LotSizeSquareFeet", "AssociationFee"]:
    cap_values[column] = df.loc[train_mask, column].quantile(0.999)
    df[column] = df[column].clip(upper=cap_values[column])

preprocessing_summary = pd.DataFrame(
    [
        {"item": "raw imported rows", "value": len(raw)},
        {"item": "CA SFR rows with positive price", "value": int(row_filter.sum())},
        {"item": "training target outliers removed", "value": len(removed_train_indices)},
        {"item": "June test rows removed", "value": 0},
        {"item": "missing-indicator source columns", "value": len(missing_indicator_sources)},
        {"item": "W3 retained/derived base features", "value": len(BASE_FEATURE_COLUMNS)},
        {"item": "low target threshold (train only)", "value": low_price_threshold},
        {"item": "price/sqft upper cutoff (train only)", "value": price_per_area_cutoff},
    ]
)
display(preprocessing_summary)
display(
    pd.DataFrame(
        {
            "rule": invalid_rule_counts.keys(),
            "values_converted_to_missing": invalid_rule_counts.values(),
        }
    )
)
display(
    pd.DataFrame(
        {
            "continuous_feature": cap_values.keys(),
            "training_q999_cap": cap_values.values(),
        }
    )
)

,item,value
0,raw imported rows,"283,176.0000"
1,CA SFR rows with positive price,"143,071.0000"
2,training target outliers removed,154.0000
3,June test rows removed,0.0000
4,missing-indicator source columns,18.0000
5,W3 retained/derived base features,34.0000
6,low target threshold (train only),"90,409.7500"
7,price/sqft upper cutoff (train only),"11,150.1819"


,rule,values_converted_to_missing
0,Latitude,27
1,Longitude,39
2,LotSizeSquareFeet,377
3,LivingArea,53
4,ParkingTotal,22
5,GarageSpaces,0
6,BathroomsTotalInteger,61
7,BedroomsTotal,77
8,MainLevelBedrooms_negative,0
9,AssociationFee,0


,continuous_feature,training_q999_cap
0,LivingArea,"10,265.2200"
1,LotSizeSquareFeet,"5,593,624.8728"
2,AssociationFee,"4,054.8902"


## Recreate the Week 6 engineered feature set

The shared predictors add intrinsic ratios, property age, cyclical sale-month
terms, logged area measures, and the target-independent CDE unified-school-
district spatial join used in W7.

In [4]:
bedrooms = pd.to_numeric(df["BedroomsTotal"], errors="coerce")
bathrooms = pd.to_numeric(df["BathroomsTotalInteger"], errors="coerce")
df["BedBathRatio"] = np.divide(
    bedrooms,
    bathrooms,
    out=np.full(len(df), np.nan, dtype=float),
    where=bathrooms.gt(0).to_numpy(),
)
df["BedBathRatio"] = df["BedBathRatio"].clip(lower=0, upper=10)

raw_property_age = (
    df["CloseDate"].dt.year
    - pd.to_numeric(df["YearBuilt"], errors="coerce")
)
df["PropertyAgeAtSale"] = raw_property_age.where(
    raw_property_age.between(0, 200)
)
df["PropertyAgeAtSale_missing_ind"] = (
    df["PropertyAgeAtSale"].isna().astype("int8")
)

sale_month_number = df["CloseDate"].dt.month
df["SaleMonthSin"] = np.sin(2 * np.pi * sale_month_number / 12)
df["SaleMonthCos"] = np.cos(2 * np.pi * sale_month_number / 12)
df["LogLivingArea"] = np.log1p(
    pd.to_numeric(df["LivingArea"], errors="coerce").clip(lower=0)
)
df["LogLotSizeSquareFeet"] = np.log1p(
    pd.to_numeric(df["LotSizeSquareFeet"], errors="coerce").clip(lower=0)
)

if not SCHOOL_DISTRICT_FILE.exists():
    raise FileNotFoundError(
        "Week 6 CDE boundary resource was not found: "
        f"{SCHOOL_DISTRICT_FILE}"
    )

districts = gpd.read_file(
    SCHOOL_DISTRICT_FILE,
    columns=["DistrictName", "DistrictType"],
).to_crs("EPSG:4326")
unified_districts = districts.loc[
    districts["DistrictType"].eq("Unified")
    & districts["DistrictName"].notna(),
    ["DistrictName", "geometry"],
].copy()
if unified_districts.empty:
    raise ValueError("No DistrictType='Unified' polygons found.")

points = gpd.GeoDataFrame(
    {"source_row_index": df.index},
    geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]),
    crs="EPSG:4326",
)
point_district_matches = gpd.sjoin(
    points,
    unified_districts,
    how="left",
    predicate="within",
)
match_counts = (
    point_district_matches.groupby("source_row_index")["DistrictName"]
    .count()
    .reindex(df.index, fill_value=0)
)
if match_counts.gt(1).any():
    raise ValueError("A property matched more than one Unified district polygon.")

district_name_by_row = (
    point_district_matches.groupby("source_row_index")["DistrictName"]
    .first()
    .reindex(df.index)
)
df["DistrictName"] = district_name_by_row
df["SchoolDistrictBoundaryMatched"] = (
    df["DistrictName"].notna().astype("int8")
)
df["DistrictName"] = df["DistrictName"].fillna("No boundary match")

feature_engineering_summary = pd.DataFrame(
    [
        {"item": "CDE polygons loaded", "value": len(districts)},
        {"item": "Unified polygons used", "value": len(unified_districts)},
        {
            "item": "properties matched to Unified district",
            "value": int(df["SchoolDistrictBoundaryMatched"].sum()),
        },
        {
            "item": "properties with no Unified match",
            "value": int((1 - df["SchoolDistrictBoundaryMatched"]).sum()),
        },
        {
            "item": "unique matched districts",
            "value": int(
                df.loc[
                    df["SchoolDistrictBoundaryMatched"].eq(1),
                    "DistrictName",
                ].nunique()
            ),
        },
        {
            "item": "new Week 6 engineered features",
            "value": 9,
        },
    ]
)
display(feature_engineering_summary)

,item,value
0,CDE polygons loaded,936
1,Unified polygons used,345
2,properties matched to Unified district,108348
3,properties with no Unified match,34569
4,unique matched districts,322
5,new Week 6 engineered features,9


## Final split, leakage audit, and common feature matrix

The notebook verifies all 12 expected training months and exactly one test
month. Trace fields, dates, source fields, and the target are excluded from the
predictor matrix. Imputation, scaling, and encoding are fit on training rows
only and then applied unchanged to the held-out month.

In [5]:
TRACE_COLUMNS = [
    "split",
    "SourceFile",
    "SourceMonth",
    "CloseDate",
    "SaleMonth",
    "ClosePrice",
]
train = df.loc[df["split"].eq("train")].copy()
test = df.loc[df["split"].eq("test")].copy()

expected_train_months = pd.period_range(
    TRAIN_START_MONTH,
    TRAIN_END_MONTH,
    freq="M",
).astype(str).tolist()
actual_train_months = sorted(train["SaleMonth"].dropna().unique().tolist())
actual_test_months = sorted(test["SaleMonth"].dropna().unique().tolist())
if actual_train_months != expected_train_months:
    raise ValueError(
        f"Expected training months {expected_train_months}, "
        f"found {actual_train_months}"
    )
if actual_test_months != [TEST_MONTH]:
    raise ValueError(
        f"Expected test month {[TEST_MONTH]}, found {actual_test_months}"
    )

feature_columns = [
    column
    for column in df.columns
    if column not in TRACE_COLUMNS
]
leakage_matches = [
    column
    for column in feature_columns
    if any(term.lower() in column.lower() for term in FORBIDDEN_FEATURE_TERMS)
]
if leakage_matches:
    raise ValueError(f"Leakage columns found: {leakage_matches}")

X_train = train[feature_columns].copy()
X_test = test[feature_columns].copy()
y_train = train["ClosePrice"].to_numpy(dtype=np.float64)
y_test = test["ClosePrice"].to_numpy(dtype=np.float64)

numeric_columns = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_columns = [
    column for column in feature_columns if column not in numeric_columns
]
for column in categorical_columns:
    X_train[column] = X_train[column].astype(object)
    X_test[column] = X_test[column].astype(object)
X_train = X_train.where(pd.notna(X_train), np.nan)
X_test = X_test.where(pd.notna(X_test), np.nan)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ]
)
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="constant", fill_value="Unknown"),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="infrequent_if_exist",
                min_frequency=50,
                dtype=np.float32,
            ),
        ),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_columns),
        ("cat", categorical_transformer, categorical_columns),
    ],
    sparse_threshold=0.3,
)

split_summary = pd.DataFrame(
    [
        {
            "split": "training",
            "months": f"{TRAIN_START_MONTH} to {TRAIN_END_MONTH}",
            "rows": len(train),
        },
        {
            "split": "untouched test",
            "months": TEST_MONTH,
            "rows": len(test),
        },
    ]
)
display(split_summary)
display(
    pd.DataFrame(
        [
            {"audit": "readable predictor columns", "value": len(feature_columns)},
            {"audit": "numeric predictor columns", "value": len(numeric_columns)},
            {
                "audit": "categorical predictor columns",
                "value": len(categorical_columns),
            },
            {"audit": "forbidden feature matches", "value": len(leakage_matches)},
            {"audit": "test rows removed", "value": 0},
        ]
    )
)

,split,months,rows
0,training,2025-06 to 2026-05,130060
1,untouched test,2026-06,12857


,audit,value
0,readable predictor columns,61
1,numeric predictor columns,53
2,categorical predictor columns,8
3,forbidden feature matches,0
4,test rows removed,0


In [6]:
def regression_metrics(y_true, y_pred):
    if np.any(y_true <= 0):
        raise ValueError("Percentage metrics require positive sale prices.")
    absolute_percentage_error = np.abs(y_true - y_pred) / y_true
    mse = float(mean_squared_error(y_true, y_pred))
    return {
        "r2": float(r2_score(y_true, y_pred)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "mape": float(np.mean(absolute_percentage_error)),
        "mdape": float(np.median(absolute_percentage_error)),
        "mse": mse,
        "rmse": float(np.sqrt(mse)),
    }


def evaluate_model(model_name, family, estimator, parameters, model_origin):
    fit_start = time.perf_counter()
    estimator.fit(X_train_processed, y_train)
    fit_seconds = time.perf_counter() - fit_start

    train_predictions = estimator.predict(X_train_processed)
    test_predictions = estimator.predict(X_test_processed)
    train_metrics = regression_metrics(y_train, train_predictions)
    test_metrics = regression_metrics(y_test, test_predictions)

    result = {
        "model": model_name,
        "family": family,
        "model_origin": model_origin,
        "train_months": f"{TRAIN_START_MONTH} to {TRAIN_END_MONTH}",
        "test_month": TEST_MONTH,
        "train_rows": len(train),
        "test_rows": len(test),
        "encoded_feature_count": X_train_processed.shape[1],
        "train_r2": train_metrics["r2"],
        "test_r2": test_metrics["r2"],
        "test_mae": test_metrics["mae"],
        "test_mape": test_metrics["mape"],
        "test_mdape": test_metrics["mdape"],
        "test_mse": test_metrics["mse"],
        "test_rmse": test_metrics["rmse"],
        "fit_seconds": fit_seconds,
        "selected_parameters": json.dumps(parameters, sort_keys=True),
    }
    return result, test_predictions


def display_test_metrics(result):
    display(
        pd.DataFrame([result])[
            [
                "model",
                "test_r2",
                "test_mape",
                "test_mdape",
                "test_mse",
                "test_rmse",
                "fit_seconds",
            ]
        ]
    )


final_preprocessor = clone(preprocessor)
preprocess_start = time.perf_counter()
X_train_processed = final_preprocessor.fit_transform(X_train)
X_test_processed = final_preprocessor.transform(X_test)
preprocess_seconds = time.perf_counter() - preprocess_start
encoded_feature_names = final_preprocessor.get_feature_names_out()

matrix_summary = pd.DataFrame(
    [
        {
            "matrix": "X_train_processed",
            "rows": X_train_processed.shape[0],
            "encoded_features": X_train_processed.shape[1],
        },
        {
            "matrix": "X_test_processed",
            "rows": X_test_processed.shape[0],
            "encoded_features": X_test_processed.shape[1],
        },
    ]
)
display(matrix_summary)

,matrix,rows,encoded_features
0,X_train_processed,130060,1193
1,X_test_processed,12857,1193


## Model 1 - Linear Regression

The Week 4 baseline is refit on the shifted 12-month training window. This
separate block records its June 2026 test metrics.

In [7]:
linear_regression_parameters = {"n_jobs": N_JOBS}
linear_regression_result, linear_regression_test_predictions = evaluate_model(
    model_name="Linear Regression",
    family="Linear Regression",
    estimator=LinearRegression(**linear_regression_parameters),
    parameters=linear_regression_parameters,
    model_origin="Week 4 baseline",
)
display_test_metrics(linear_regression_result)

,model,test_r2,test_mape,test_mdape,test_mse,test_rmse,fit_seconds
0,Linear Regression,0.6986,0.3382,0.1798,"711,534,617,310.2603","843,525.1136",26.8734


## Model 2 - Decision Tree

The Week 6 decision-tree setting (depth 24, minimum leaf size 10) is refit and
evaluated independently on the same processed matrices.

In [8]:
decision_tree_parameters = {
    "max_depth": 24,
    "min_samples_leaf": 10,
    "random_state": RANDOM_STATE,
}
decision_tree_result, decision_tree_test_predictions = evaluate_model(
    model_name="Decision Tree (depth 24, leaf 10)",
    family="Decision Tree",
    estimator=DecisionTreeRegressor(**decision_tree_parameters),
    parameters=decision_tree_parameters,
    model_origin="Week 6 selected setting",
)
display_test_metrics(decision_tree_result)

,model,test_r2,test_mape,test_mdape,test_mse,test_rmse,fit_seconds
0,"Decision Tree (depth 24, leaf 10)",0.7216,0.2630,0.1061,"657,192,451,938.3954","810,674.0726",24.2353


## Model 3 - Random Forest

The W7 reference configuration is refit here so its metrics are produced inside
this notebook rather than copied from a prior CSV.

In [9]:
random_forest_parameters = {
    "n_estimators": 60,
    "max_depth": 30,
    "min_samples_leaf": 10,
    "max_features": 0.7,
    "random_state": RANDOM_STATE,
    "n_jobs": N_JOBS,
}
random_forest_result, random_forest_test_predictions = evaluate_model(
    model_name="Random Forest (60 trees, depth 30, leaf 10, max_features 0.7)",
    family="Random Forest",
    estimator=RandomForestRegressor(**random_forest_parameters),
    parameters=random_forest_parameters,
    model_origin="Week 6 selected setting / W7 reference",
)
display_test_metrics(random_forest_result)

,model,test_r2,test_mape,test_mdape,test_mse,test_rmse,fit_seconds
0,"Random Forest (60 trees, depth 30, leaf 10, max_features 0.7)",0.7486,0.2369,0.0914,"593,492,333,606.8525","770,384.5362",325.6162


## Model 4 - XGBoost

The W7 selected XGBoost setting is refit without additional tuning and
evaluated in its own block.

In [10]:
xgboost_parameters = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "subsample": 0.85,
    "colsample_bytree": 0.80,
    "reg_lambda": 5.0,
    "random_state": RANDOM_STATE,
    "n_jobs": N_JOBS,
    "verbosity": 0,
    "max_depth": 8,
    "learning_rate": 0.05,
    "n_estimators": 500,
}
xgboost_result, xgboost_test_predictions = evaluate_model(
    model_name="XGBoost (depth 8, lr 0.05, 500 trees)",
    family="XGBoost",
    estimator=xgb.XGBRegressor(**xgboost_parameters),
    parameters=xgboost_parameters,
    model_origin="Week 7 selected setting",
)
display_test_metrics(xgboost_result)

,model,test_r2,test_mape,test_mdape,test_mse,test_rmse,fit_seconds
0,"XGBoost (depth 8, lr 0.05, 500 trees)",0.8116,0.2520,0.1177,"444,759,032,847.8608","666,902.5662",13.1481


## Model 5 - LightGBM

The W7 selected LightGBM setting is refit without additional tuning and
evaluated in its own block.

In [11]:
lightgbm_parameters = {
    "objective": "regression",
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.80,
    "reg_lambda": 5.0,
    "min_child_samples": 30,
    "random_state": RANDOM_STATE,
    "n_jobs": N_JOBS,
    "verbosity": -1,
    "max_depth": 10,
    "num_leaves": 63,
    "learning_rate": 0.05,
    "n_estimators": 500,
}
lightgbm_result, lightgbm_test_predictions = evaluate_model(
    model_name="LightGBM (depth 10, lr 0.05, 500 trees)",
    family="LightGBM",
    estimator=lgb.LGBMRegressor(**lightgbm_parameters),
    parameters=lightgbm_parameters,
    model_origin="Week 7 selected setting",
)
display_test_metrics(lightgbm_result)

,model,test_r2,test_mape,test_mdape,test_mse,test_rmse,fit_seconds
0,"LightGBM (depth 10, lr 0.05, 500 trees)",0.7975,0.2656,0.1223,"477,979,240,725.1068","691,360.4275",8.7810


## Consolidated five-model metrics

Summary of all five models from prior weeks

In [12]:
metrics_summary = pd.DataFrame(
    [
        linear_regression_result,
        decision_tree_result,
        random_forest_result,
        xgboost_result,
        lightgbm_result,
    ]
)

column_order = [
    "model",
    "family",
    "model_origin",
    "train_months",
    "test_month",
    "train_rows",
    "test_rows",
    "encoded_feature_count",
    "train_r2",
    "test_r2",
    "test_mae",
    "test_mape",
    "test_mdape",
    "test_mse",
    "test_rmse",
    "fit_seconds",
    "selected_parameters",
]
metrics_summary = metrics_summary[column_order].sort_values(
    ["test_r2", "test_rmse"],
    ascending=[False, True],
).reset_index(drop=True)

metrics_path = OUTPUT_DIR / "metrics_summary.csv"
metrics_summary.to_csv(metrics_path, index=False)

required_metric_columns = [
    "test_r2",
    "test_mape",
    "test_mdape",
    "test_mse",
    "test_rmse",
]
assert len(metrics_summary) == 5
assert metrics_summary["family"].nunique() == 5
assert np.isfinite(metrics_summary[required_metric_columns].to_numpy()).all()
assert metrics_summary["train_months"].eq("2025-06 to 2026-05").all()
assert metrics_summary["test_month"].eq("2026-06").all()
assert metrics_summary["train_rows"].eq(len(train)).all()
assert metrics_summary["test_rows"].eq(len(test)).all()
assert np.allclose(
    metrics_summary["test_mse"],
    metrics_summary["test_rmse"] ** 2,
)

display(
    metrics_summary[
        [
            "model",
            "test_r2",
            "test_mape",
            "test_mdape",
            "test_mse",
            "test_rmse",
        ]
    ].style.format(
        {
            "test_r2": "{:.3f}",
            "test_mape": "{:.2%}",
            "test_mdape": "{:.2%}",
            "test_mse": "${:,.0f}",
            "test_rmse": "${:,.0f}",
        }
    )
)
print(f"Saved {metrics_path}")

,model,test_r2,test_mape,test_mdape,test_mse,test_rmse
0,"XGBoost (depth 8, lr 0.05, 500 trees)",0.812,25.20%,11.77%,"$444,759,032,848","$666,903"
1,"LightGBM (depth 10, lr 0.05, 500 trees)",0.798,26.56%,12.23%,"$477,979,240,725","$691,360"
2,"Random Forest (60 trees, depth 30, leaf 10, max_features 0.7)",0.749,23.69%,9.14%,"$593,492,333,607","$770,385"
3,"Decision Tree (depth 24, leaf 10)",0.722,26.30%,10.61%,"$657,192,451,938","$810,674"
4,Linear Regression,0.699,33.82%,17.98%,"$711,534,617,310","$843,525"


Saved /Users/HP/Documents/IDX Exchange/W8 Evaluation Expansion/metrics_summary.csv


## Reading the metrics

- **R²** measures the share of test-month price variance explained; higher is
  better.
- **MAPE** is the mean absolute percentage error and can be pulled upward by
  difficult or extreme cases (e.g. upper outliers/luxury home prices); lower is better.
- **MdAPE** is the median absolute percentage error and describes the typical
  sale more robustly; lower is better.
- **MSE** averages squared dollar errors, while **RMSE** returns that penalty to
  dollar units. Both emphasize large misses; lower is better.

Because model rankings can differ by metric, the summary should be read as a
trade-off between statewide variance/tail-error control (R², MSE, RMSE) and
typical relative accuracy (MdAPE).

## Price-band evaluation across all five models

June 2026 sales are divided into five equal-frequency bands using actual test
prices. These are diagnostic groups only: the band cutoffs are created after
prediction and do not affect model fitting or selection.

Performance within each model is ranked primarily by **MdAPE**, because it
compares typical relative error across differently priced homes. MAPE, MAE,
MSE, RMSE, and within-band R² are also retained. Within-band R² can be negative
because each band intentionally has a narrow range of actual prices, so it
should not be used alone to decide which band performs best.

In [13]:
model_band_inputs = [
    {
        "model": linear_regression_result["model"],
        "family": linear_regression_result["family"],
        "predictions": linear_regression_test_predictions,
    },
    {
        "model": decision_tree_result["model"],
        "family": decision_tree_result["family"],
        "predictions": decision_tree_test_predictions,
    },
    {
        "model": random_forest_result["model"],
        "family": random_forest_result["family"],
        "predictions": random_forest_test_predictions,
    },
    {
        "model": xgboost_result["model"],
        "family": xgboost_result["family"],
        "predictions": xgboost_test_predictions,
    },
    {
        "model": lightgbm_result["model"],
        "family": lightgbm_result["family"],
        "predictions": lightgbm_test_predictions,
    },
]

test_band_codes, price_band_edges = pd.qcut(
    pd.Series(y_test),
    q=5,
    labels=False,
    retbins=True,
    duplicates="drop",
)
test_band_codes = test_band_codes.to_numpy(dtype=int)
if len(price_band_edges) != 6:
    raise ValueError(
        f"Expected five price bands, found {len(price_band_edges) - 1}."
    )


def compact_price(value):
    if value >= 1_000_000:
        text = f"${value / 1_000_000:.2f}M"
        return text.replace(".00M", "M").replace("0M", "M")
    return f"${value / 1_000:.0f}k"


price_band_labels = [
    f"Up to {compact_price(price_band_edges[1])}",
    *[
        f"{compact_price(price_band_edges[index])}-"
        f"{compact_price(price_band_edges[index + 1])}"
        for index in range(1, len(price_band_edges) - 2)
    ],
    f"{compact_price(price_band_edges[-2])}+",
]
if len(price_band_labels) != 5:
    raise ValueError(f"Unexpected price-band labels: {price_band_labels}")

model_order = {
    item["family"]: order
    for order, item in enumerate(model_band_inputs, start=1)
}
price_band_rows = []
for item in model_band_inputs:
    predictions = np.asarray(item["predictions"], dtype=float)
    if predictions.shape != y_test.shape:
        raise ValueError(
            f"Prediction shape mismatch for {item['family']}: "
            f"{predictions.shape} versus {y_test.shape}"
        )

    for band_code, band_label in enumerate(price_band_labels):
        band_mask = test_band_codes == band_code
        band_actual = y_test[band_mask]
        band_predictions = predictions[band_mask]
        band_metrics = regression_metrics(band_actual, band_predictions)
        price_band_rows.append(
            {
                "model": item["model"],
                "family": item["family"],
                "model_order": model_order[item["family"]],
                "band_order": band_code + 1,
                "price_band": band_label,
                "lower_bound": float(price_band_edges[band_code]),
                "upper_bound": float(price_band_edges[band_code + 1]),
                "test_rows": int(band_mask.sum()),
                "mean_close_price": float(np.mean(band_actual)),
                "median_close_price": float(np.median(band_actual)),
                "r2_within_band": band_metrics["r2"],
                "mae": band_metrics["mae"],
                "mape": band_metrics["mape"],
                "mdape": band_metrics["mdape"],
                "mse": band_metrics["mse"],
                "rmse": band_metrics["rmse"],
            }
        )

price_band_metrics = pd.DataFrame(price_band_rows).sort_values(
    ["model_order", "band_order"]
).reset_index(drop=True)
for metric in ["mdape", "mape", "rmse"]:
    price_band_metrics[f"{metric}_rank_within_model"] = (
        price_band_metrics.groupby("family")[metric]
        .rank(method="first", ascending=True)
        .astype(int)
    )

highlight_rows = []
for item in model_band_inputs:
    family_rows = price_band_metrics.loc[
        price_band_metrics["family"].eq(item["family"])
    ].sort_values(["mdape", "mape", "rmse"])
    best = family_rows.iloc[0]
    second = family_rows.iloc[1]
    weakest = family_rows.iloc[-1]
    highlight_rows.append(
        {
            "model": item["model"],
            "family": item["family"],
            "best_price_band": best["price_band"],
            "best_band_mdape": best["mdape"],
            "best_band_mape": best["mape"],
            "best_band_rmse": best["rmse"],
            "second_best_price_band": second["price_band"],
            "second_best_band_mdape": second["mdape"],
            "weakest_price_band": weakest["price_band"],
            "weakest_band_mdape": weakest["mdape"],
        }
    )
price_band_highlights = pd.DataFrame(highlight_rows)

price_band_metrics_path = OUTPUT_DIR / "price_band_metrics.csv"
price_band_highlights_path = OUTPUT_DIR / "price_band_highlights.csv"
price_band_metrics.drop(columns="model_order").to_csv(
    price_band_metrics_path,
    index=False,
)
price_band_highlights.to_csv(price_band_highlights_path, index=False)

assert len(price_band_metrics) == 25
assert price_band_metrics["family"].nunique() == 5
assert price_band_metrics["price_band"].nunique() == 5
assert (
    price_band_metrics.groupby("family")["test_rows"].sum().eq(len(test)).all()
)
assert np.isfinite(
    price_band_metrics[
        ["r2_within_band", "mae", "mape", "mdape", "mse", "rmse"]
    ].to_numpy()
).all()
assert price_band_metrics.groupby("family")["mdape_rank_within_model"].apply(
    lambda ranks: sorted(ranks.tolist()) == [1, 2, 3, 4, 5]
).all()

display(
    price_band_metrics[
        [
            "family",
            "price_band",
            "test_rows",
            "r2_within_band",
            "mape",
            "mdape",
            "rmse",
            "mdape_rank_within_model",
        ]
    ].style.format(
        {
            "r2_within_band": "{:.3f}",
            "mape": "{:.2%}",
            "mdape": "{:.2%}",
            "rmse": "${:,.0f}",
        }
    )
)
display(
    price_band_highlights.style.format(
        {
            "best_band_mdape": "{:.2%}",
            "best_band_mape": "{:.2%}",
            "best_band_rmse": "${:,.0f}",
            "second_best_band_mdape": "{:.2%}",
            "weakest_band_mdape": "{:.2%}",
        }
    )
)
print(f"Saved {price_band_metrics_path}")
print(f"Saved {price_band_highlights_path}")

,family,price_band,test_rows,r2_within_band,mape,mdape,rmse,mdape_rank_within_model
0,Linear Regression,Up to $575k,2616,-4.327,77.09%,27.65%,"$240,899",5
1,Linear Regression,$575k-$800k,2546,-21.941,27.02%,19.57%,"$314,990",4
2,Linear Regression,$800k-$1.1M,2561,-13.816,21.85%,15.47%,"$326,643",2
3,Linear Regression,$1.1M-$1.67M,2563,-5.751,19.73%,13.78%,"$415,237",1
4,Linear Regression,$1.67M+,2571,0.565,22.50%,16.64%,"$1,767,133",3
5,Decision Tree,Up to $575k,2616,-0.516,62.45%,9.51%,"$128,526",3
6,Decision Tree,$575k-$800k,2546,-5.283,12.90%,7.90%,"$164,846",1
7,Decision Tree,$800k-$1.1M,2561,-14.209,15.17%,9.50%,"$330,944",2
8,Decision Tree,$1.1M-$1.67M,2563,-4.650,17.23%,11.87%,"$379,893",4
9,Decision Tree,$1.67M+,2571,0.583,22.93%,16.64%,"$1,729,100",5


,model,family,best_price_band,best_band_mdape,best_band_mape,best_band_rmse,second_best_price_band,second_best_band_mdape,weakest_price_band,weakest_band_mdape
0,Linear Regression,Linear Regression,$1.1M-$1.67M,13.78%,19.73%,"$415,237",$800k-$1.1M,15.47%,Up to $575k,27.65%
1,"Decision Tree (depth 24, leaf 10)",Decision Tree,$575k-$800k,7.90%,12.90%,"$164,846",$800k-$1.1M,9.50%,$1.67M+,16.64%
2,"Random Forest (60 trees, depth 30, leaf 10, max_features 0.7)",Random Forest,$575k-$800k,6.24%,11.07%,"$144,719",$800k-$1.1M,8.20%,$1.67M+,14.10%
3,"XGBoost (depth 8, lr 0.05, 500 trees)",XGBoost,$800k-$1.1M,9.88%,14.14%,"$284,230",$575k-$800k,10.10%,Up to $575k,15.06%
4,"LightGBM (depth 10, lr 0.05, 500 trees)",LightGBM,$800k-$1.1M,10.34%,14.72%,"$247,475",$575k-$800k,10.48%,Up to $575k,14.98%


Saved /Users/HP/Documents/IDX Exchange/W8 Evaluation Expansion/price_band_metrics.csv
Saved /Users/HP/Documents/IDX Exchange/W8 Evaluation Expansion/price_band_highlights.csv


## Reading the price-band results

- Rank 1 identifies each model's lowest held-out MdAPE and therefore its best
  band for a typical sale on a relative-error basis.
- Dollar errors generally increase with sale price, so RMSE should be compared
  alongside—not substituted for—the relative-error ranks.
- The lowest band can show elevated MAPE even when MdAPE is moderate because a
  small number of large relative misses pull the mean upward.
- The highest band is open-ended and includes luxury outliers, which can raise
  RMSE and MSE substantially even when typical percentage error remains
  competitive.